Data Dictionary

Project: Smart Agriculture and Climate Resilience 
File described:`data/clean/agri_dataset.csv` | Grain: one row per country, crop and year (1990-2022)

Sources:FAOSTAT Crops and Livestock Products (QCL), FAO, CC BY 4.0 (yield and area), and NASA POWER monthly AG-community data (climate). Built by `build_dataset.py`.

This notebook does two things:
1. Writes the dictionary (name, type, meaning, units/categories, expected missing values) for all 14 columns.
2. Checks every claim in the dictionary against the real data, so the document can't drift away from the file.

1. LOAD THE DATA

In [1]:
from pathlib import Path
import pandas as pd

from pathlib import Path
import pandas as pd

data_path = Path("agri_dataset.csv")
df = pd.read_csv(
    data_path,
    parse_dates=["observation_date"]
)

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)

display(df.head())

Dataset loaded successfully!
Rows and columns: (36664, 14)


,record_id,observation_date,region,crop,rainfall_mm,temperature_c,humidity_pct,soil_moisture,yield_kg,plot_area_ha,country,iso3,year,yield_kg_ha
0,1,1990-01-01,Southern Asia,Barley,165.00,7.280,41.915,0.487,216000000.0,205000.0,Afghanistan,AFG,1990,1053.66
1,2,1991-01-01,Southern Asia,Barley,218.00,6.275,47.304,0.498,217000000.0,204000.0,Afghanistan,AFG,1991,1063.73
2,3,1992-01-01,Southern Asia,Barley,241.31,5.786,47.861,0.505,200000000.0,250000.0,Afghanistan,AFG,1992,800.00
3,4,1993-01-01,Southern Asia,Barley,187.41,6.414,43.721,0.492,250000000.0,250000.0,Afghanistan,AFG,1993,1000.00
4,5,1994-01-01,Southern Asia,Barley,194.99,6.823,44.446,0.486,230000000.0,250000.0,Afghanistan,AFG,1994,920.00


 2. THE DICTIONARY

In [18]:
dictionary = pd.DataFrame([
    (1,  "record_id",        "int",      "Row identifier, 1 to 36,664, assigned after sorting by country, crop, year.",
         "Unique integer", "No"),
    (2,  "observation_date", "datetime", "1 January of `year`. Derived from `year`, not a real survey date.",
         "1990-01-01 to 2022-01-01", "No"),
    (3,  "region",           "string",   "UN sub-region of the country (assigned with country_converter).",
         "22 categories, e.g. Eastern Africa, Southern Asia", "No"),
    (4,  "crop",             "string",   "Crop name.",
         "10 categories: Barley, Cassava, Maize, Millet, Potatoes, Rice, Sorghum, Soybeans, Sweet potatoes, Wheat", "No"),
    (5,  "rainfall_mm",      "float",    "Total rainfall for the year at the country centroid (NASA POWER PRECTOTCORR, mm/day x days in month, summed over 12 months).",
         "mm per year", "No (years without 12 valid months were dropped in the build)"),
    (6,  "temperature_c",    "float",    "Mean annual air temperature at 2 m (NASA POWER T2M) at the country centroid.",
         "degrees Celsius", "No"),
    (7,  "humidity_pct",     "float",    "Mean annual relative humidity at 2 m (NASA POWER RH2M) at the country centroid.",
         "percent, 0 to 100", "No"),
    (8,  "soil_moisture",    "float",    "Mean annual root-zone soil wetness (NASA POWER GWETROOT) at the country centroid.",
         "index, 0 (dry) to 1 (saturated)", "No"),
    (9,  "yield_kg",         "float",    "Total national production of the crop that year (FAOSTAT Production, tonnes converted to kg). Not a single farm.",
         "kilograms", "No (rows with missing or zero production were dropped)"),
    (10, "plot_area_ha",     "float",    "Total national area harvested for the crop that year (FAOSTAT Area harvested). Not a single plot.",
         "hectares", "No (rows with missing or zero area were dropped)"),
    (11, "country",          "string",   "Country name as written by FAOSTAT.",
         "197 names, including historical ones (Czechoslovakia, Ethiopia PDR, Sudan (former))", "No"),
    (12, "iso3",             "string",   "ISO 3166-1 alpha-3 country code.",
         "194 codes. CZE, ETH and SDN each cover two country names", "No"),
    (13, "year",             "int",      "Calendar year of the observation.",
         "1990 to 2022", "No"),
    (14, "yield_kg_ha",      "float",    "Yield per hectare = yield_kg / plot_area_ha, rounded to 2 decimals.",
         "kg per hectare", "No (derived from yield_kg and plot_area_ha)"),
], columns=["#", "column", "type", "meaning", "units / categories", "missing expected?"]).set_index("#")

dictionary

,column,type,meaning,units / categories,missing expected?
#,,,,,
1,record_id,int,"Row identifier, 1 to 36,664, assigned after sorting by country, crop, year.",Unique integer,No
2,observation_date,datetime,"1 January of `year`. Derived from `year`, not a real survey date.",1990-01-01 to 2022-01-01,No
3,region,string,UN sub-region of the country (assigned with country_converter).,"22 categories, e.g. Eastern Africa, Southern Asia",No
4,crop,string,Crop name.,"10 categories: Barley, Cassava, Maize, Millet, Potatoes, Rice, Sorghum, Soybeans, Sweet potatoes, Wheat",No
5,rainfall_mm,float,"Total rainfall for the year at the country centroid (NASA POWER PRECTOTCORR, mm/day x days in month, summed over 12 months).",mm per year,No (years without 12 valid months were dropped in the build)
6,temperature_c,float,Mean annual air temperature at 2 m (NASA POWER T2M) at the country centroid.,degrees Celsius,No
7,humidity_pct,float,Mean annual relative humidity at 2 m (NASA POWER RH2M) at the country centroid.,"percent, 0 to 100",No
8,soil_moisture,float,Mean annual root-zone soil wetness (NASA POWER GWETROOT) at the country centroid.,"index, 0 (dry) to 1 (saturated)",No
9,yield_kg,float,"Total national production of the crop that year (FAOSTAT Production, tonnes converted to kg). Not a single farm.",kilograms,No (rows with missing or zero production were dropped)


3. CHECK THE DICTIONARY AGAINST THE COLUMN

For each column we read the real dtype, null count, number of unique values and the min/max. Then we compare them with what the dictionary says.

In [2]:
from pathlib import Path
import pandas as pd

# Load the dataset from Downloads
data_path = Path("agri_dataset.csv")

df = pd.read_csv(
    data_path,
    parse_dates=["observation_date"]
)

# Check every column
rows = []

for col in df.columns:
    s = df[col]

    is_num = (
        pd.api.types.is_numeric_dtype(s)
        or pd.api.types.is_datetime64_any_dtype(s)
    )

    rows.append({
        "column": col,
        "actual dtype": str(s.dtype),
        "nulls": int(s.isna().sum()),
        "unique": int(s.nunique()),
        "min": s.min() if is_num else "",
        "max": s.max() if is_num else ""
    })

# Display the results
observed = pd.DataFrame(rows).set_index("column")

display(observed)

,actual dtype,nulls,unique,min,max
column,,,,,
record_id,int64,0,36664,1,36664
observation_date,datetime64[us],0,33,1990-01-01 00:00:00,2022-01-01 00:00:00
region,str,0,22,,
crop,str,0,10,,
rainfall_mm,float64,0,6073,0.0,6989.04
temperature_c,float64,0,5130,-8.023,30.497
humidity_pct,float64,0,5752,18.284,95.175
soil_moisture,float64,0,809,0.09,0.99
yield_kg,float64,0,29445,200.0,412262180000.0


In [16]:
# Every dictionary claim, tested. A failed check raises an error.
checks = {
    "14 columns documented, same order as file":   list(dictionary["column"]) == list(df.columns),
    "no nulls anywhere (all 'missing: No')":       df.isna().sum().sum() == 0,
    "record_id unique, 1..36,664":                 df["record_id"].is_unique and df["record_id"].min() == 1 and df["record_id"].max() == len(df),
    "observation_date is always 1 Jan of year":    ((df["observation_date"].dt.month == 1) & (df["observation_date"].dt.day == 1) & (df["observation_date"].dt.year == df["year"])).all(),
    "22 regions":                                  df["region"].nunique() == 22,
    "10 crops":                                    df["crop"].nunique() == 10,
    "humidity_pct within 0-100":                   df["humidity_pct"].between(0, 100).all(),
    "soil_moisture within 0-1":                    df["soil_moisture"].between(0, 1).all(),
    "rainfall_mm >= 0":                            (df["rainfall_mm"] >= 0).all(),
    "yield_kg and plot_area_ha > 0":               ((df["yield_kg"] > 0) & (df["plot_area_ha"] > 0)).all(),
    "197 country names":                           df["country"].nunique() == 197,
    "194 ISO3 codes":                              df["iso3"].nunique() == 194,
    "years 1990 to 2022":                          (df["year"].min(), df["year"].max()) == (1990, 2022),
    "yield_kg_ha = yield_kg / plot_area_ha (+-0.01)": ((df["yield_kg"] / df["plot_area_ha"] - df["yield_kg_ha"]).abs() <= 0.01).all(),
}
result = pd.DataFrame({"check": list(checks), "result": ["PASS" if bool(v) else "FAIL" for v in checks.values()]})
assert (result["result"] == "PASS").all(), "The dictionary no longer matches the data"
result

,check,result
0,"14 columns documented, same order as file",PASS
1,no nulls anywhere (all 'missing: No'),PASS
2,"record_id unique, 1..36,664",PASS
3,observation_date is always 1 Jan of year,PASS
4,22 regions,PASS
5,10 crops,PASS
6,humidity_pct within 0-100,PASS
7,soil_moisture within 0-1,PASS
8,rainfall_mm >= 0,PASS
9,yield_kg and plot_area_ha > 0,PASS


 4. Categories in the dictionary

In [17]:
pd.concat({
    "crop": df["crop"].value_counts(),
    "region": df["region"].value_counts(),
}, axis=0).rename("rows").to_frame()

rows
crop   Maize                      5167
       Potatoes                   4963
       Wheat                      3953
       Rice                       3765
       Sweet potatoes             3532
       Sorghum                    3400
       Barley                     3220
       Cassava                    3133
       Soybeans                   2986
       Millet                     2545
region Eastern Africa             4176
       Western Africa             3537
       South America              3112
       Western Asia               2988
       Southern Europe            2220
       Eastern Europe             2181
       Southern Asia              2094
       South-eastern Asia         1982
       Central America            1909
       Middle Africa              1849
       Caribbean                  1792
       Northern Africa            1308
       Eastern Asia               1287
       Central Asia               1109
       Western Europe             1108
       Northern Europe            1001
       Southern Africa             949
       Melanesia                   699
       Australia and New Zealand   462
       Northern America            462
       Polynesia                   295
       Micronesia                  144

5. Notes for anyone using this data

- Climate repeats across crops.Columns 5 to 8 are one value per country-year, so every crop in that country and year shares them. Rows are not fully independent.
- Totals, not farms. `yield_kg`, `plot_area_ha` and `yield_kg_ha` are national figures. Don't draw conclusions about single farms.
- Skewed values.** `yield_kg`, `plot_area_ha` and `yield_kg_ha` are heavily right-skewed. Use medians or a log scale.
- Compare within a crop.** Median yield runs from about 1,000 kg/ha (Millet) to about 16,100 kg/ha (Potatoes).
- Flagged values. 23 rows have `yield_kg_ha` below 1 or above 50,000. 3 ISO3 codes cover two country names each. Both are flagged in `02_exploration.ipynb` and handled in Week 3.